In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# @title 1. TPU環境構築とライブラリのインストール
# Keras 3 と KerasNLP をインストール（JAXバックエンド用）
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

# JAXをバックエンドに指定（TPUの性能を最大化するため）
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # メモリをフル活用

import keras
import keras_nlp
import jax
import numpy as np

# --- TPUの検出と初期化 ---
print("🚀 TPU初期化プロセス開始...")
try:
    # TPUデバイスの確認
    tpu = jax.devices()
    print(f"✅ TPU検出成功: {len(tpu)} コアが利用可能です。")
    print(f"   デバイス詳細: {tpu}")
except:
    print("⚠️ TPUが見つかりません。ランタイムの設定を確認してください。CPU/GPUで動作します。")

# 混合精度演算の設定（計算速度向上とメモリ節約）
keras.mixed_precision.set_global_policy("mixed_bfloat16")
print("⚡ Mixed Precision (bfloat16) を有効化しました。")

In [ ]:
# @title 2. Gemmaモデルのロードと「思考する学習ループ」の準備
# 軽量な2Bモデルを使用 (colab TPUで余裕を持って動くサイズ)
MODEL_ID = "gemma2_2b_en" 

print(f"\n📥 {MODEL_ID} をロード中... (JAX用にコンパイルされます)")
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_ID)
print("✅ モデルロード完了")

# --- 詳細実況のためのカスタムコールバック ---
class ThoughtProcessLogger(keras.callbacks.Callback):
    """
    学習の1ステップごとに、TPU内部の挙動や思考過程（Lossの変化）を
    詳細に実況するコールバック
    """
    def on_train_begin(self, logs=None):
        print("\n🤖 [AI Teacher] ファインチューニングを開始します。")
        print("   ここからの目標は、汎用的なGemmaモデルを、特定のデータセットに特化させることです。")
        print("   TPUの全コア(8コア)を使って、データ並列処理で勾配を計算します。\n")

    def on_epoch_begin(self, epoch, logs=None):
        print(f"📅 [Epoch {epoch + 1}] 開始")
        print("   データをTPUメモリに分散転送中...")

    def on_train_batch_end(self, batch, logs=None):
        # 5バッチごとに詳細な思考ログを出力
        if batch % 5 == 0:
            loss = logs['loss']
            # 擬似的な内部状態の解説
            print(f"\n   🔍 [Step {batch}] ----------------------------------------")
            print(f"   📉 Current Loss: {loss:.4f}")
            
            if loss > 2.0:
                print("      👉 まだ誤差が大きいです。モデルは入力と出力の関係を模索中。")
                print("      👉 Backward Pass: 勾配が大きく変動しており、重みが大きく更新されています。")
            elif loss > 1.0:
                print("      👉 誤差が縮まってきました。文法やパターンを掴み始めています。")
                print("      👉 Optimizer: 学習率に従い、パラメータの微調整フェーズに入りつつあります。")
            else:
                print("      👉 非常に低いLossです！モデルはデータセットの特徴をほぼ完全に捉えました。")
            
            print("      💾 [Hardware] TPU Matrix Units (MXU) Utilization: High")
            print("   --------------------------------------------------------")

# --- LoRA (Low-Rank Adaptation) の設定 ---
# 全パラメータを学習すると重すぎるため、LoRAで効率化します
print("\n🔧 LoRA (Low-Rank Adaptation) を適用中...")
print("   説明: 巨大な行列を直接更新せず、低ランク行列の積として近似更新します。")
print("   効果: 学習可能なパラメータ数を劇的に（1/100以下に）削減します。")

gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.preprocessor.sequence_length = 512 # シーケンス長

# 学習対象のパラメータ数を表示
gemma_lm.summary()

In [ ]:
# @title 3. データセット準備とファインチューニング実行
# デモ用のデータセット（JSON形式などで本来は用意するが、ここではリストで作成）
# 例えば「AIアシスタントとしての振る舞い」を教えるデータ
data = [
    "User: Hello, who are you? \nModel: I am Gemma, an AI assistant developed by Google.",
    "User: What is TPU? \nModel: TPU stands for Tensor Processing Unit, an AI accelerator application-specific integrated circuit (ASIC).",
    "User: Explain normalization. \nModel: Normalization is a technique to scale input data to a specific range, often improving convergence speed.",
    "User: Python code for loop. \nModel: for i in range(10): print(i)",
    # データを複製してバッチ数を稼ぐ
] * 20 

print(f"\n📚 学習データ: {len(data)} 件のサンプルを準備しました。")

# オプティマイザの設定 (AdamW)
# JAX環境ではコンパイル時に最適化されるため高速です
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=["accuracy"],
)

print("\n🚀 ファインチューニングを実行します（実況ログ付き）...")

# 学習開始
# ThoughtProcessLoggerにより、学習経過がリアルタイムで「解説」されます
gemma_lm.fit(
    data, 
    epochs=1, 
    batch_size=4,
    callbacks=[ThoughtProcessLogger()]
)

print("\n✅ 学習完了。LoRAウェイトが更新されました。")

# --- 推論テスト ---
print("\n🧪 [Inference Test] 学習後のモデルで生成テスト:")
prompt = "User: What is TPU? \nModel:"
print(f"Input: {prompt}")

# 生成
generated = gemma_lm.generate(prompt, max_length=64)
print(f"Output: {generated}")